# Week 1 · Day 4 — Lab 3
## Leaving the Notebook: Modules, Tests, and the Pipeline Hand-off

> **AI Engineering Academy** · Gamut Technology Services

Notebooks are for *exploration*. The moment logic stabilizes - you're copying cells
between notebooks, another step needs to call it, or it's complex enough to deserve
tests - it belongs in a **tested, importable module**. This lab makes that transition
concrete: extract the fetch logic from Lab 1 into a `client.py` module, **unit-test it
by mocking HTTP** with the `responses` library (no network, fully deterministic), and
hand clean data off as **Parquet** for Week 2.

### ⚙️ No server needed here
Unlike Labs 1-2, this lab makes **no live calls**. That's the whole point of mocking:
`responses` intercepts requests so your tests are fast and don't depend on a running
API. This is *why* you extract logic to modules - you cannot `pytest` a notebook cell.

### Learning objectives
1. Extract stabilized notebook logic into an importable `.py` **module** (`%%writefile` + `autoreload`).
2. **Unit-test** the fetch function with `responses` - single page, pagination, and error handling.
3. Assert **exact request counts** to prove pagination stops correctly.
4. Save a cleaned DataFrame to **Parquet** and verify the round-trip for the Week 2 hand-off.

### Time budget — ~88 min
| Segment | Time |
|---|---|
| Setup & the module workflow | 6 min |
| **A.** Extract logic to a module | 14 min |
| **B.** Unit-test a single page | 14 min |
| **C.** Test pagination (+ call count) | 16 min |
| **D.** Test error handling | 14 min |
| **E.** Save clean data as Parquet | 20 min |
| Wrap-up + stretch | 4 min |

### Files you need (beside this notebook)
- Nothing external - this lab **writes** `client.py`, `validate.py`, and `test_client.py` itself with `%%writefile`.
- `responses`, `pandas`, and `pyarrow` must be installed (see `requirements.txt`).


In [ ]:
%pip install --upgrade responses pandas pyarrow

In [ ]:
# --- Setup: the module-development workflow --------------------------------
# This lab needs NO live server: we test HTTP code by MOCKING it with the
# `responses` library (fast, deterministic, offline). We also extract stable
# logic into real .py modules and hand clean data off as Parquet.
%load_ext autoreload
%autoreload 2

import os, json
import requests
import responses
import pandas as pd
from pathlib import Path

os.environ.setdefault("API_KEY", "local-dev-key")   # module reads this if present

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready. pandas", pd.__version__)

## Part A — Extract logic to a module  *(guided)*

The fetch loop you wrote inline in Lab 1 has stabilized. Move it into `client.py` so
any script, DAG, or test can import it. `%%writefile` writes the cell to disk;
`%autoreload 2` (set above) reloads the module automatically whenever it changes.


In [ ]:
%%writefile client.py
"""client.py - stabilized data-acquisition logic, extracted from the notebook."""
import os
import logging

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logger = logging.getLogger(__name__)


def build_session(api_key: str | None = None) -> requests.Session:
    """Return a Session with auth, connection pooling, and retry/backoff."""
    api_key = api_key or os.environ.get("API_KEY", "")
    session = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=0,                 # 0 in tests for speed; use 1+ in production
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=True,             # exhausted retries -> RetryError
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    if api_key:
        session.headers.update({"Authorization": f"Bearer {api_key}", "Accept": "application/json"})
    return session


def fetch_all_records(base_url: str, page_size: int = 100) -> list[dict]:
    """Fetch all paginated records from base_url.

    Walks page/per_page pagination until a short page signals the end.

    Args:
        base_url: The API endpoint URL.
        page_size: Records requested per page.

    Returns:
        The complete list of record dicts.
    """
    records: list[dict] = []
    page = 1
    logger.info("Starting fetch from %s", base_url)
    with build_session() as session:
        while True:
            resp = session.get(base_url, params={"page": page, "per_page": page_size}, timeout=(5, 30))
            resp.raise_for_status()
            batch = resp.json()["data"]
            records.extend(batch)
            if len(batch) < page_size:
                break
            page += 1
    logger.info("Fetch complete. Total records: %d", len(records))
    return records

### Exercise A1 — Import it and confirm it's real
Import `fetch_all_records` and `build_session` from `client`. Set `is_callable` to
whether `fetch_all_records` is callable, and `has_doc` to whether it has a non-empty
docstring. (Being importable and documented is exactly what makes it testable - which
you'll do next.)


💡 **Hint.** `from client import fetch_all_records, build_session`. `callable(fn)`
and `bool(fn.__doc__)` give the two booleans.


In [ ]:
# TODO: import fetch_all_records and build_session from client
is_callable = None
has_doc = None

In [ ]:
check("A1: fetch_all_records is importable and callable", lambda: is_callable is True)
check("A1: the function has a docstring", lambda: has_doc is True)

## Part B — Unit-test a single page with `responses`

`responses` intercepts outgoing `requests` calls and returns canned responses - so
tests run offline, instantly, and deterministically. Decorate a test with
`@responses.activate`, register the fake endpoint with `responses.add(...)`, then call
your function and assert on the result.


In [ ]:
@responses.activate
def demo_single():
    responses.add(
        responses.GET, "https://api.test/v1/records",
        json={"data": [{"id": 1, "title": "Test"}]}, status=200,
    )
    result = fetch_all_records("https://api.test/v1/records")
    assert result == [{"id": 1, "title": "Test"}]
    return True

print("demo_single passed:", demo_single())

### Exercise B1 — Your first mocked test
Write `test_single_page()` decorated with `@responses.activate`. Register a `GET` on
`https://api.example.com/v1/records` returning
`{"data": [{"id": 10}, {"id": 11}]}` with status 200. Call `fetch_all_records` on that
URL and `assert` the result equals `[{"id": 10}, {"id": 11}]`. Return `True`. Then
call it into `b1_passed`.


💡 **Hint.** Mirror `demo_single`. Because the mocked page has only 2 records (fewer
than the default `page_size=100`), the fetch loop stops after one request.


In [ ]:
@responses.activate
def test_single_page():
    # TODO: responses.add(...) a 200 GET returning two records
    # TODO: call fetch_all_records and assert the result
    return True

b1_passed = None     # TODO: call test_single_page()

In [ ]:
check("B1: single-page test passes", lambda: b1_passed is True)

## Part C — Test pagination (and prove the request count)

The real risk in pagination is *silent truncation* or an *extra request*. A good test
mocks multiple pages and asserts both the assembled data **and** the exact number of
HTTP calls. `responses` serves queued responses in order and records every call in
`responses.calls`.


### Exercise C1 — Two pages, then stop
Write `test_paginates()` (`@responses.activate`). Queue **two** GETs on
`https://api.example.com/v1/records`: the first returns a full page of 100 records
(`[{"id": i} for i in range(100)]`), the second a partial page of 40
(`range(100, 140)`). Call `fetch_all_records`, then assert: the result has **140**
records, the first id is `0` and the last is `139`, and **exactly 2** requests were
made (`len(responses.calls) == 2`). Return `True` into `c1_passed`.


💡 **Hint.** Call `responses.add(...)` twice - `responses` serves them in order for
repeated calls to the same URL. The partial second page (40 < 100) is what stops the
loop. `len(responses.calls)` is the total intercepted requests.


In [ ]:
@responses.activate
def test_paginates():
    base = "https://api.example.com/v1/records"
    # TODO: add page 1 (100 records) and page 2 (40 records)
    # TODO: call fetch_all_records; assert length, first/last id, and call count
    return True

c1_passed = None     # TODO: call test_paginates()

In [ ]:
check("C1: pagination test passes (140 records, 2 calls)", lambda: c1_passed is True)

## Part D — Test error handling

Your fetch function is configured to retry on 5xx and then raise. Prove it: mock a
persistent `500` and assert the call raises rather than returning bad data. (With
`raise_on_status=True`, exhausted retries surface as `requests.exceptions.RetryError`,
a subclass of `RequestException`.)


### Exercise D1 — Assert it raises on a server error
Write `test_raises_on_500()` (`@responses.activate`). Register a `GET` on
`https://api.example.com/v1/records` returning status `500`. Call `fetch_all_records`
inside a `try/except requests.exceptions.RequestException`; return `True` if the
exception is raised, `False` if it isn't. Store the result in `d1_passed`.


💡 **Hint.** `responses.add(responses.GET, url, status=500)`. Wrap the call:
`try: fetch_all_records(url); return False` /
`except requests.exceptions.RequestException: return True`.


In [ ]:
@responses.activate
def test_raises_on_500():
    url = "https://api.example.com/v1/records"
    # TODO: register a 500 response
    # TODO: return True iff fetch_all_records raises a RequestException
    return False

d1_passed = None     # TODO: call test_raises_on_500()

In [ ]:
check("D1: fetch raises on a persistent 500", lambda: d1_passed is True)

## Part E — Save clean data as Parquet (the Week 2 hand-off)

Once data is fetched and cleaned, persist it as **Parquet**: columnar, compressed,
and schema-preserving. This is the artifact Week 2's pipeline consumes. Here you'll
mock a fetch, build and lightly clean a DataFrame, save it, and verify the round-trip.


In [ ]:
@responses.activate
def fetch_sample_df():
    """Mock a small fetch and return a cleaned DataFrame."""
    responses.add(
        responses.GET, "https://api.example.com/v1/records",
        json={"data": [
            {"id": 1, "title": "A", "created_at": "2024-01-01T00:00:00", "score": 91.5},
            {"id": 2, "title": "B", "created_at": "2024-01-02T00:00:00", "score": 88.0},
            {"id": 2, "title": "B", "created_at": "2024-01-02T00:00:00", "score": 88.0},  # dup
        ]}, status=200,
    )
    recs = fetch_all_records("https://api.example.com/v1/records")
    frame = pd.DataFrame(recs).drop_duplicates()
    frame["created_at"] = pd.to_datetime(frame["created_at"])
    return frame

clean_df = fetch_sample_df()
print(clean_df)
print(clean_df.dtypes)

### Exercise E1 — `save_for_pipeline` and verify the round-trip
Write `save_for_pipeline(df, name, out_dir="data/processed")` that creates `out_dir`
(with `parents=True, exist_ok=True`), writes `df` to `{out_dir}/{name}.parquet`
(`index=False`, `engine="pyarrow"`), and returns the `Path`. Save `clean_df` as
`"records_v1"`, reload it into `reloaded`, and set `roundtrips` to whether `reloaded`
equals `clean_df` (dtypes and values intact).


💡 **Hint.** Use `pathlib.Path`. `df.to_parquet(path, index=False, engine="pyarrow")`.
Reload with `pd.read_parquet(path)`. Compare with `reloaded.equals(clean_df)` - the
datetime and float dtypes survive Parquet exactly.


In [ ]:
def save_for_pipeline(df, name, out_dir="data/processed"):
    ...  # TODO: mkdir, write parquet, return the Path
    return None

saved_path = None    # TODO: save clean_df as "records_v1"
reloaded = None      # TODO: read it back
roundtrips = None    # TODO: does reloaded equal clean_df?

In [ ]:
check("E1: parquet file was written", lambda: saved_path is not None and Path(saved_path).exists())
check("E1: de-duplicated to 2 rows before saving", lambda: len(clean_df) == 2)
check("E1: round-trip preserved dtypes and values", lambda: roundtrips is True)
check("E1: created_at survived as datetime",
      lambda: str(reloaded["created_at"].dtype).startswith("datetime64"))

## Stretch goals *(for fast finishers)*

**S1 — Extract a validator module.** The cell below `%%writefile`s a `validate.py`
with `validate_columns(df, required_cols)`. Import it and confirm it **passes** on
`clean_df` with `{"id", "title", "score"}` and **raises** `ValueError` on a missing
column.

**S2 — A `pytest`-ready test file.** The cell below `%%writefile`s a `test_client.py`
holding your single-page and pagination tests. In a real project you'd run
`pytest test_client.py -q`; here just confirm the file exists and is non-empty.


In [ ]:
%%writefile validate.py
"""validate.py - lightweight data guards, extracted for reuse and testing."""
import pandas as pd


def validate_columns(df: pd.DataFrame, required_cols: set) -> pd.DataFrame:
    """Raise ValueError if any required column is missing; else return df."""
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return df

In [ ]:
%%writefile test_client.py
"""test_client.py - run in a real project with:  pytest test_client.py -q"""
import responses
from client import fetch_all_records


@responses.activate
def test_single_page():
    responses.add(
        responses.GET, "https://api.example.com/v1/records",
        json={"data": [{"id": 10}, {"id": 11}]}, status=200,
    )
    assert fetch_all_records("https://api.example.com/v1/records") == [{"id": 10}, {"id": 11}]


@responses.activate
def test_paginates():
    base = "https://api.example.com/v1/records"
    responses.add(responses.GET, base, json={"data": [{"id": i} for i in range(100)]}, status=200)
    responses.add(responses.GET, base, json={"data": [{"id": i} for i in range(100, 140)]}, status=200)
    result = fetch_all_records(base)
    assert len(result) == 140
    assert len(responses.calls) == 2

In [ ]:
# S1 — import the validator you just wrote and exercise it
val_ok = None        # TODO: validate_columns(clean_df, {"id","title","score"}) returns a df (not None)
val_raised = None    # TODO: True if it raises ValueError on a missing column

# S2 — confirm the pytest-ready file exists and is non-empty
test_file_ok = None  # TODO: Path("test_client.py") exists and has size > 0

In [ ]:
check("S1: validator passes on present columns", lambda: val_ok is True)
check("S1: validator raises on a missing column", lambda: val_raised is True)
check("S2: pytest-ready test_client.py was written", lambda: test_file_ok is True)

## Wrap-up — what you can now do

- Extract stabilized notebook logic into an importable, documented module.
- Unit-test HTTP code offline with `responses` - single page, pagination, and errors.
- Assert exact request counts to prove a pagination loop is correct.
- Hand clean data off as Parquet with a verified round-trip.

**That's Day 4.** You can pull data reliably from a REST API (`requests`: timeouts,
`raise_for_status`, sessions, retries, pagination, secrets), explore it in Jupyter
without hidden-state traps, and graduate stable logic into tested modules with a
Parquet hand-off - the exact loop every later week builds on.
